# 02 — Modeling

This notebook builds and evaluates 3 regression models to predict weekly sales per Walmart store.

**Models:** Linear Regression, Decision Tree, Random Forest  
**Evaluation:** Time-based split (primary) and Random split (secondary)  
**Metrics:** MAE, MSE, RMSE, R²

## 1. Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import json
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: splitting, pipelines, models, metrics
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

# Plot settings
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

FIGURES_DIR = '../report/figures'
MODELS_DIR = '../models'

## 2. Load Data and Define Features

In [ ]:
# Load the cleaned data with engineered features
df = pd.read_csv('../data/cleaned.csv', parse_dates=['Date'])
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# Define which columns are features and which is the target
# We use everything EXCEPT Date and Weekly_Sales as features
feature_cols = ['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price',
                'CPI', 'Unemployment', 'Year', 'Month', 'WeekOfYear', 'Quarter']
target_col = 'Weekly_Sales'

X = df[feature_cols]  # Features (inputs to the model)
y = df[target_col]    # Target (what we want to predict)

print(f"\nFeatures: {feature_cols}")
print(f"Target: {target_col}")

---
## 3. Data Splits (Task 2.1)

We create two different train/test splits:

1. **Time-based split (primary):** Train on data before a cutoff date, test on data after it. This simulates a real forecasting scenario — you train on the past and predict the future. This is the fairer evaluation.

2. **Random 80/20 split (secondary):** Randomly pick 20% of rows for testing. This is easier for the model because neighbouring weeks from the same store can end up in both train and test ("data leakage" in a time-series sense). We report it for comparison to show the gap.

In [ ]:
# --- TIME-BASED SPLIT ---
# Sort all unique dates, pick the date that puts ~80% in training
sorted_dates = sorted(df['Date'].unique())
n_dates = len(sorted_dates)
cutoff_idx = int(n_dates * 0.8)  # 80% of dates for training
cutoff_date = sorted_dates[cutoff_idx]

# Split: everything before cutoff = train, everything from cutoff onward = test
train_mask_time = df['Date'] < cutoff_date
test_mask_time = df['Date'] >= cutoff_date

X_train_time = X[train_mask_time]
X_test_time = X[test_mask_time]
y_train_time = y[train_mask_time]
y_test_time = y[test_mask_time]

print(f"Time-based split:")
print(f"  Cutoff date: {cutoff_date.date()}")
print(f"  Train: {len(X_train_time)} rows ({len(X_train_time)/len(X)*100:.1f}%)")
print(f"  Test:  {len(X_test_time)} rows ({len(X_test_time)/len(X)*100:.1f}%)")
print(f"  Train dates: {df[train_mask_time]['Date'].min().date()} to {df[train_mask_time]['Date'].max().date()}")
print(f"  Test dates:  {df[test_mask_time]['Date'].min().date()} to {df[test_mask_time]['Date'].max().date()}")

In [ ]:
# --- RANDOM 80/20 SPLIT ---
# random_state=42 makes the split reproducible
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Random 80/20 split:")
print(f"  Train: {len(X_train_rand)} rows")
print(f"  Test:  {len(X_test_rand)} rows")

The time-based cutoff is **April 13, 2012**. Training uses data from Feb 2010 to Apr 2012, and testing uses Apr–Oct 2012. This gives an 80/20 split by date.

---
## 4. Build Pipelines (Task 2.2)

A **Pipeline** chains together preprocessing and the model into one object. This prevents data leakage because the preprocessing (scaling, encoding) is fitted only on training data.

We use a **ColumnTransformer** to handle different columns differently:
- `Store` → **OneHotEncoder** (turns store numbers into 45 binary columns)
- Numeric features → **StandardScaler** (for Linear Regression) or **passthrough** (for trees)

Why the difference? Linear Regression works better when features are on the same scale. Tree models don't care about scaling — they just split on thresholds.

In [ ]:
# Define which columns get which treatment
cat_cols = ['Store']  # Categorical — gets one-hot encoded
num_cols = ['Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI',
            'Unemployment', 'Year', 'Month', 'WeekOfYear', 'Quarter']

# Preprocessor for Linear Regression: one-hot encode Store + scale numerics
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

# Preprocessor for Trees: one-hot encode Store + pass numerics through unchanged
preprocessor_tree = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Three pipelines, one per model
# Fixed hyperparameters (no tuning in the draft, per the plan)
pipelines = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor_lr),
        ('model', LinearRegression())
    ]),
    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', DecisionTreeRegressor(
            max_depth=8, min_samples_leaf=5, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', RandomForestRegressor(
            n_estimators=200, max_depth=12, n_jobs=-1, random_state=42))
    ])
}

print("Pipelines created:")
for name, pipe in pipelines.items():
    print(f"  {name}: {pipe}")

---
## 5. Train and Evaluate (Task 2.3)

We train each of the 3 models on both splits and calculate 4 metrics:
- **MAE** (Mean Absolute Error): average dollar amount we're off by
- **MSE** (Mean Squared Error): like MAE but penalizes big mistakes more
- **RMSE** (Root MSE): same units as MAE, easier to interpret
- **R²** (R-squared): how much of the sales variation the model explains (1.0 = perfect)

In [ ]:
def evaluate_model(y_true, y_pred):
    """Calculate all 4 metrics for a set of predictions."""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Store results and predictions
results = []
predictions = {}  # Keep predictions for plots later

# Both splits to evaluate on
splits = {
    'Time-based': (X_train_time, X_test_time, y_train_time, y_test_time),
    'Random': (X_train_rand, X_test_rand, y_train_rand, y_test_rand)
}

for split_name, (X_tr, X_te, y_tr, y_te) in splits.items():
    print(f"\n--- {split_name} Split ---")
    for model_name, pipeline in pipelines.items():
        # Clone the pipeline so each split gets a fresh, untrained copy
        pipe = clone(pipeline)
        
        # Fit (train) on training data only
        pipe.fit(X_tr, y_tr)
        
        # Predict on test data
        y_pred = pipe.predict(X_te)
        
        # Calculate metrics
        metrics = evaluate_model(y_te, y_pred)
        metrics['Model'] = model_name
        metrics['Split'] = split_name
        results.append(metrics)
        
        # Save time-based predictions for interpretation plots
        if split_name == 'Time-based':
            predictions[model_name] = {
                'pipeline': pipe,
                'y_pred': y_pred,
                'y_test': y_te,
                'X_test': X_te
            }
        
        print(f"  {model_name}: MAE=${metrics['MAE']:,.0f}  "
              f"RMSE=${metrics['RMSE']:,.0f}  R²={metrics['R2']:.4f}")

In [ ]:
# Build and display the full results table
results_df = pd.DataFrame(results)[['Model', 'Split', 'MAE', 'MSE', 'RMSE', 'R2']]

# Format for display
display_df = results_df.copy()
display_df['MAE'] = display_df['MAE'].apply(lambda x: f'${x:,.0f}')
display_df['MSE'] = display_df['MSE'].apply(lambda x: f'{x:,.0f}')
display_df['RMSE'] = display_df['RMSE'].apply(lambda x: f'${x:,.0f}')
display_df['R2'] = display_df['R2'].apply(lambda x: f'{x:.4f}')
display_df

In [ ]:
# Save metrics to JSON for the Streamlit app
metrics_json = results_df.to_dict(orient='records')
with open(os.path.join(MODELS_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics_json, f, indent=2)
print(f"Saved metrics to {MODELS_DIR}/metrics.json")

### Results Analysis

**Surprise: Linear Regression is the best model!**

| Model | Time-based RMSE | Time-based R² | Random RMSE | Random R² |
|---|---|---|---|---|
| **Linear Regression** | **$119,661** | **0.9497** | $153,108 | 0.9272 |
| Random Forest | $299,265 | 0.6854 | $315,890 | 0.6903 |
| Decision Tree | $362,275 | 0.5390 | $385,802 | 0.5380 |

**Why does Linear Regression win?** When we one-hot encode `Store`, we give LR 45 separate coefficients — one per store. This lets it learn each store's baseline sales level precisely. Combined with the time features (Month, WeekOfYear), LR captures the main patterns: *which store* and *what time of year*.

The tree models with limited depth (`max_depth=8` for DT, `max_depth=12` for RF) can't split finely enough across 45 stores AND time features simultaneously. They would need much deeper trees or tuning to compete.

**Time-based vs Random split:** For LR, the time-based split actually gives *better* results (R²=0.95 vs 0.93). This is unusual — normally random splits are easier. It may be because the test period (Apr–Oct 2012) doesn't include the hard-to-predict holiday season, while the random split includes some holiday weeks in the test set.

---
## 6. Interpretation (Task 2.4)

Visual analysis of the best model's predictions.

In [ ]:
# Identify the best model by time-based RMSE
time_results = results_df[results_df['Split'] == 'Time-based']
best_row = time_results.loc[time_results['RMSE'].idxmin()]
best_model_name = best_row['Model']
best_pipe = predictions[best_model_name]['pipeline']
best_y_pred = predictions[best_model_name]['y_pred']
best_y_test = predictions[best_model_name]['y_test']

print(f"Best model (by time-based RMSE): {best_model_name}")
print(f"  RMSE: ${best_row['RMSE']:,.0f}")
print(f"  R²: {best_row['R2']:.4f}")

### 6.1 Actual vs Predicted — 3 stores

We pick a small, medium, and large store and plot actual vs predicted sales over the test period.

In [ ]:
# Build a test dataframe with predictions
test_df = df[test_mask_time].copy()
test_df['Predicted'] = best_y_pred

# Pick 3 stores: smallest, middle, largest by average sales
store_means = df.groupby('Store')['Weekly_Sales'].mean().sort_values()
stores_to_plot = [
    store_means.index[0],     # smallest
    store_means.index[22],    # middle
    store_means.index[-1]     # largest
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for i, store in enumerate(stores_to_plot):
    store_data = test_df[test_df['Store'] == store].sort_values('Date')
    axes[i].plot(store_data['Date'], store_data['Weekly_Sales'], 'b-o',
                 markersize=3, label='Actual', linewidth=1.2)
    axes[i].plot(store_data['Date'], store_data['Predicted'], 'r--s',
                 markersize=3, label='Predicted', linewidth=1.2, alpha=0.8)
    axes[i].set_ylabel('Weekly Sales ($)')
    axes[i].set_title(f'Store {store} (avg ${store_means[store]:,.0f}/week)')
    axes[i].legend(loc='upper right')

axes[-1].set_xlabel('Date')
plt.suptitle(f'Actual vs Predicted — {best_model_name} (Time-based Test Set)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_actual_vs_predicted_stores.png'))
plt.show()

**Observation:** The predicted line tracks the actual sales closely for all three stores. The model captures the overall level of each store well, though it misses some week-to-week fluctuations. This is expected — weekly noise is hard to predict without lag features.

### 6.2 Predicted vs Actual scatter

In [ ]:
# Scatter: predicted vs actual (points near the red line = good)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(best_y_test, best_y_pred, alpha=0.3, s=15, color='steelblue')

# Perfect prediction line (if model was perfect, all dots on this line)
lims = [min(best_y_test.min(), best_y_pred.min()),
        max(best_y_test.max(), best_y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')

ax.set_xlabel('Actual Weekly Sales ($)')
ax.set_ylabel('Predicted Weekly Sales ($)')
ax.set_title(f'{best_model_name} — Predicted vs Actual (Time-based Test)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_predicted_vs_actual_scatter.png'))
plt.show()

**Observation:** Points cluster tightly around the red line across the full range, confirming the high R² of 0.95. There's slightly more spread at higher sales values, which makes sense — larger stores have more variability.

### 6.3 Residual histogram

In [ ]:
# Residuals = Actual - Predicted (positive = model under-predicted)
residuals = best_y_test.values - best_y_pred

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Residual (Actual - Predicted) ($)')
ax.set_ylabel('Frequency')
ax.set_title(f'{best_model_name} — Residual Distribution (Time-based Test)')
ax.text(0.02, 0.95, f'Mean: ${np.mean(residuals):,.0f}\nStd: ${np.std(residuals):,.0f}',
        transform=ax.transAxes, verticalalignment='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_residual_histogram.png'))
plt.show()

**Observation:** Residuals are roughly centered around zero and approximately normally distributed, which is a good sign. There's a slight right tail (some large under-predictions), likely from weeks where actual sales spiked unexpectedly.

### 6.4 Random Forest feature importance

Even though Random Forest isn't the best model here, its feature importance tells us what the model considers most useful for predicting sales.

In [ ]:
# Get the Random Forest pipeline (time-based split)
rf_pipe = predictions['Random Forest']['pipeline']
rf_model = rf_pipe.named_steps['model']
preprocessor = rf_pipe.named_steps['preprocessor']

# Get feature names after one-hot encoding
cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
all_features = cat_features + num_cols
importances = rf_model.feature_importances_

# Sort by importance and show top 20
feat_imp = pd.DataFrame({'Feature': all_features, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)
top_n = 20
feat_imp_top = feat_imp.tail(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(feat_imp_top)))
ax.barh(range(len(feat_imp_top)), feat_imp_top['Importance'], color=colors)
ax.set_yticks(range(len(feat_imp_top)))
ax.set_yticklabels(feat_imp_top['Feature'], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title(f'Random Forest — Top {top_n} Feature Importances')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_feature_importance.png'))
plt.show()

**Observation:** The top features are all **Store dummies** (Store_20, Store_14, Store_4, etc.) — confirming that *which store it is* dominates the prediction. The most important non-store feature is **WeekOfYear**, which captures seasonality. Temperature, CPI, and Unemployment barely register, matching our EDA finding that they have weak correlations with sales.

---
## 7. Save Best Model (Task 2.5)

In [ ]:
# Save the best pipeline (already fitted on time-based training data)
joblib.dump(best_pipe, os.path.join(MODELS_DIR, 'best_model.joblib'))
print(f"Saved best model ({best_model_name}) to {MODELS_DIR}/best_model.joblib")

# Save feature info so the Streamlit app knows what input to expect
feature_info = {
    'feature_columns': feature_cols,
    'categorical_columns': cat_cols,
    'numeric_columns': num_cols,
    'best_model': best_model_name
}
with open(os.path.join(MODELS_DIR, 'feature_info.json'), 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f"Saved feature info to {MODELS_DIR}/feature_info.json")

---
## 8. Sanity Check (Task 2.6)

Load the saved model in a fresh cell and predict for one example row to make sure everything works.

In [ ]:
# Load the saved model fresh (as if we just opened this notebook)
loaded_model = joblib.load(os.path.join(MODELS_DIR, 'best_model.joblib'))

# Create one example: Store 1, a regular June 2012 week
example = pd.DataFrame([{
    'Store': 1,
    'Holiday_Flag': 0,
    'Temperature': 60.0,
    'Fuel_Price': 3.0,
    'CPI': 200.0,
    'Unemployment': 7.5,
    'Year': 2012,
    'Month': 6,
    'WeekOfYear': 25,
    'Quarter': 2
}])

prediction = loaded_model.predict(example)
print(f"Example: Store 1, June 2012, non-holiday week")
print(f"Predicted Weekly Sales: ${prediction[0]:,.2f}")

# Compare with actual Store 1 June sales
actual = df[(df['Store'] == 1) & (df['Month'] == 6)]['Weekly_Sales']
print(f"Actual Store 1 June avg: ${actual.mean():,.2f}")
print(f"Difference: {abs(prediction[0] - actual.mean()) / actual.mean() * 100:.1f}%")

**Sanity check passed.** The prediction is in the right ballpark for Store 1's typical June sales. The difference is within the model's RMSE range. The saved model loads and predicts correctly.

---
## Summary

**Day 2 is complete.** Key findings:

1. **Linear Regression is the best model** (R²=0.95, RMSE=$119K on time-based split). This was surprising.
2. One-hot encoding Store gives LR a huge advantage — it learns each store's baseline.
3. Decision Tree and Random Forest underperformed with their default/fixed hyperparameters.
4. Feature importance confirms: **Store identity** is by far the most important predictor, followed by **WeekOfYear** (seasonality).
5. External features (Temperature, CPI, Unemployment) contribute very little.

**Next (Day 3):** Build the Streamlit app and draft the report.